In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

# FLenQA probe directions through static J-Lens concepts

For each layer, this notebook asks which vocabulary concepts are most affected by moving the hidden state in the trained probe direction. It uses the saved probes and static J-Lens only; there are no interventions or drift analyses.

In [ ]:
import json

import pandas as pd
import torch
import transformers
import jlens

from experiments.jlens_readout_sanity.constants import LENS_PATH, MODEL_PATH
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
asset_dir = context.checkpoints_dir / "flenqa-probe-assets"
probe_path = asset_dir / "probes.pt"
metadata_path = asset_dir / "metadata.json"
result_path = context.runs_dir / "flenqa-probe-jlens" / "concept_scores.parquet"

checkpoint = torch.load(probe_path, map_location="cpu", weights_only=False)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
causal_lm.eval()

num_layers = int(causal_lm.config.num_hidden_layers)
hidden_dim = int(causal_lm.config.hidden_size)
assert metadata["num_layers"] == num_layers
assert metadata["hidden_dim"] == hidden_dim

lens = jlens.JacobianLens.from_pretrained(LENS_PATH)
unembedding = causal_lm.get_output_embeddings().weight.detach().float().cpu()
assert unembedding.ndim == 2 and unembedding.shape[1] == hidden_dim

available_layers = sorted(set(lens.source_layers) & set(checkpoint["layers"]))
assert available_layers
assert all(
    lens.jacobians[layer].ndim == 2
    and lens.jacobians[layer].shape == (hidden_dim, hidden_dim)
    for layer in available_layers
)
print({"layers": available_layers, "vocab_size": unembedding.shape[0], "hidden_dim": hidden_dim})

## Compute vocabulary concept scores

The saved J-Lens convention is confirmed by `jlens_vector`: one token direction is `J_l.T @ W_U[token]`. Therefore the equivalent batched score is `W_U @ (J_l @ w_hat)`, with no normalization after the J-Lens projection.

In [ ]:
TOP_K = 15


def decode_token(tokenizer, token_id):
    raw_token = tokenizer.convert_ids_to_tokens(int(token_id))
    token = tokenizer.decode([int(token_id)], clean_up_tokenization_spaces=False)
    return raw_token, token


def concept_rows_for_layer(layer, probe_asset, jacobian, unembedding, tokenizer, top_k):
    probe_direction = probe_asset["unit_weight"].float().cpu()
    assert probe_direction.shape == (unembedding.shape[1],)
    assert torch.isfinite(probe_direction).all()
    probe_direction = probe_direction / torch.linalg.vector_norm(probe_direction)

    # J_l and W_U are [hidden, hidden] and [vocab, hidden].
    # Do not normalize J_l @ probe_direction: its magnitude is part of the score.
    concept_scores = unembedding @ (jacobian @ probe_direction)
    assert concept_scores.shape == (unembedding.shape[0],)
    assert torch.isfinite(concept_scores).all()

    rows = []
    for direction, ranked_ids in (
        ("positive", torch.topk(concept_scores, top_k).indices),
        ("negative", torch.topk(-concept_scores, top_k).indices),
    ):
        for rank, token_id in enumerate(ranked_ids.tolist(), start=1):
            raw_token, token = decode_token(tokenizer, token_id)
            rows.append(
                {
                    "layer": int(layer),
                    "token": token,
                    "raw_token": raw_token,
                    "token_id": int(token_id),
                    "score": float(concept_scores[token_id]),
                    "rank": rank,
                    "direction": direction,
                }
            )
    return rows


concept_rows = []
for layer in available_layers:
    jacobian = lens.jacobians[layer].detach().float().cpu()
    concept_rows.extend(
        concept_rows_for_layer(
            layer, checkpoint["layers"][layer], jacobian, unembedding, tokenizer, TOP_K
        )
    )
concept_results = pd.DataFrame(concept_rows)
assert set(concept_results.columns) == {
    "layer", "token", "raw_token", "token_id", "score", "rank", "direction"
}
assert len(concept_results) == len(available_layers) * 2 * TOP_K

## Top concepts by layer

Positive and negative rows retain the raw tokenizer token and its decoded display form.

In [ ]:
display(
    concept_results.sort_values(["layer", "direction", "rank"])[
        ["layer", "direction", "rank", "token", "raw_token", "token_id", "score"]
    ].round({"score": 4})
)

In [ ]:
result_path.parent.mkdir(parents=True, exist_ok=True)
concept_results.to_parquet(result_path, index=False)
print(f"Saved {len(concept_results):,} rows to {result_path}")

## Short cross-layer summary

This table only summarizes recurrence among the extracted top positive/negative concepts; it does not add an interpretation.

In [ ]:
summary = (
    concept_results.assign(abs_score=concept_results["score"].abs())
    .groupby(["token_id", "token", "raw_token"], as_index=False)
    .agg(
        layers=("layer", "nunique"),
        max_abs_score=("abs_score", "max"),
        mean_abs_score=("abs_score", "mean"),
    )
    .sort_values(["layers", "mean_abs_score"], ascending=[False, False])
    .head(20)
)
display(summary.round({"max_abs_score": 4, "mean_abs_score": 4}))